# Person 2 — Step 6: Temporal Thumbnail Analysis

How do thumbnail features evolve over time in the two groups (**independent** vs **institutional**)?

Sources (all from notebook 5 outputs):
- `advanced_thumbnail_synthesis.csv` — VLM semantic features (text function, human presence, clickbait, archetype) + DINOv2 cluster
- `opencv_features.csv` — low-level visual aesthetics (brightness, contrast, saturation, colorfulness, edge density)
- `face_canvas_ratio.csv` — OpenCV Haar-cascade face measurements
- `thumbnail_manifest.csv` — `published_at` timestamps

Pipeline:
1. Merge all sources on `video_id` and bin by publication year.
2. Track continuous VLM features (sensationalism, human-presence) over time.
3. Track OpenCV visual aesthetics (brightness, contrast, saturation, colourfulness) over time.
4. Track face metrics from both VLM and OpenCV face detector over time.
5. Track text presence and placement evolution over time.
6. Categorical share evolution per year (archetype, text function, visual style).
7. Group-divergence heatmap across all features.
8. Statistical test: early (2012–2016) vs late (2019–2025) period shift.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

sys.path.append('../../')
from shared.config import PERSON2_DIR

FIGURES = PERSON2_DIR / 'outputs' / 'figures'
TABLES  = PERSON2_DIR / 'outputs' / 'tables'
MANIFEST = PERSON2_DIR / 'outputs' / 'thumbnail_manifest.csv'

PALETTE = {'independent': '#2ecc71', 'institutional': '#3498db'}
YEAR_MIN, YEAR_MAX = 2012, 2025
EARLY = (YEAR_MIN, 2016)
LATE  = (2019, YEAR_MAX)

## 1 — Load & merge

In [ ]:
manifest = pd.read_csv(MANIFEST, parse_dates=['published_at'])
manifest['year'] = manifest['published_at'].dt.year

synth   = pd.read_csv(TABLES / 'advanced_thumbnail_synthesis.csv')
opencv  = pd.read_csv(TABLES / 'opencv_features.csv')
face_df = pd.read_csv(TABLES / 'face_canvas_ratio.csv')

df = (
    synth
    .merge(manifest[['video_id', 'published_at', 'year', 'view_count', 'like_count']], on='video_id', how='inner')
    .merge(opencv[['video_id', 'brightness', 'contrast', 'saturation', 'colorfulness', 'edge_density', 'dominant_hue']], on='video_id', how='left')
    .merge(face_df[['video_id', 'face_canvas_ratio', 'n_faces_detected']].rename(columns={
        'face_canvas_ratio': 'opencv_face_ratio',
        'n_faces_detected':  'opencv_n_faces'
    }), on='video_id', how='left')
)

df = df[(df['year'] >= YEAR_MIN) & (df['year'] <= YEAR_MAX)].copy()
df['group_label'] = df['group_label'].str.strip().str.lower()

# Cast booleans stored as strings
for col in ['hp_is_present', 'cb_has_red_arrow', 'cb_has_exag_contrast']:
    df[col] = df[col].map({True: 1, False: 0, 'True': 1, 'False': 0, 1: 1, 0: 0}).fillna(np.nan)

# Derive text-presence flag: 1 if thumbnail has any embedded text
df['has_text'] = df['embedded_text'].apply(
    lambda x: 0 if (pd.isna(x) or str(x).strip().lower() in ('', 'none', 'nan')) else 1
).astype(float)

# OpenCV face present flag
df['opencv_has_face'] = (df['opencv_n_faces'] > 0).astype(float)

print(f"Rows after merge & filter: {len(df)}")
print(df.groupby(['group_label', 'year']).size().unstack(fill_value=0).to_string())

## 2 — VLM semantic feature trends

Yearly mean ± 95 % CI for sensationalism, human-presence, and face metrics from the VLM analysis.

In [ ]:
def yearly_mean_ci(group_df, col):
    rows = []
    for year, g in group_df.groupby('year'):
        vals = g[col].dropna()
        if len(vals) < 3:
            continue
        m = vals.mean()
        se = stats.sem(vals)
        lo, hi = stats.t.interval(0.95, df=len(vals)-1, loc=m, scale=se)
        rows.append({'year': year, 'mean': m, 'lo': lo, 'hi': hi, 'n': len(vals)})
    return pd.DataFrame(rows)


VLM_CONTINUOUS = {
    'cb_sensationalism_score': 'Sensationalism score (1–5)',
    'hp_canvas_coverage_pct':  'VLM human-presence canvas %',
    'hp_is_present':           'Share with human present (VLM)',
    'hp_count':                'Avg humans per thumbnail (VLM)',
}

fig, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)
fig.suptitle('VLM semantic feature trends over time', fontsize=14, fontweight='bold')

for ax, (col, label) in zip(axes.flat, VLM_CONTINUOUS.items()):
    for grp, color in PALETTE.items():
        sub = df[df['group_label'] == grp]
        ts = yearly_mean_ci(sub, col)
        if ts.empty:
            continue
        ax.plot(ts['year'], ts['mean'], marker='o', color=color, label=grp.capitalize(), linewidth=2, markersize=5)
        ax.fill_between(ts['year'], ts['lo'], ts['hi'], color=color, alpha=0.15)
    ax.set_title(label, fontsize=11)
    ax.set_xlabel('Year')
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.legend(framealpha=0.4)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

fig.savefig(FIGURES / 'temporal_vlm_features.png', dpi=150, bbox_inches='tight')
plt.show()

## 3 — OpenCV visual aesthetics over time

Low-level perceptual properties extracted by OpenCV for every thumbnail: brightness, contrast, saturation, colourfulness, and edge density.

In [ ]:
OPENCV_FEATURES = {
    'brightness':   'Brightness (mean pixel)',
    'contrast':     'Contrast (normalised)',
    'saturation':   'Saturation (HSV mean)',
    'colorfulness': 'Colourfulness (Hasler-Süsstrunk)',
    'edge_density': 'Edge density (Canny)',
}

fig, axes = plt.subplots(2, 3, figsize=(17, 9), constrained_layout=True)
fig.suptitle('OpenCV visual aesthetics over time', fontsize=14, fontweight='bold')

for ax, (col, label) in zip(axes.flat, OPENCV_FEATURES.items()):
    for grp, color in PALETTE.items():
        sub = df[df['group_label'] == grp]
        ts = yearly_mean_ci(sub, col)
        if ts.empty:
            continue
        ax.plot(ts['year'], ts['mean'], marker='o', color=color, label=grp.capitalize(), linewidth=2, markersize=5)
        ax.fill_between(ts['year'], ts['lo'], ts['hi'], color=color, alpha=0.15)
    ax.set_title(label, fontsize=11)
    ax.set_xlabel('Year')
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.legend(framealpha=0.4)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

# hide unused 6th panel
axes.flat[-1].set_visible(False)

fig.savefig(FIGURES / 'temporal_opencv_aesthetics.png', dpi=150, bbox_inches='tight')
plt.show()

## 4 — Face metrics over time: VLM vs OpenCV

Compare two independent face signals:
- **VLM** (semantic): `hp_is_present`, `hp_canvas_coverage_pct` — answers: is there a human and how prominent?
- **OpenCV** (structural): `opencv_has_face`, `opencv_face_ratio` — answers: did Haar-cascade find a face?

Consistent trends across both methods strengthen interpretation.

In [ ]:
FACE_FEATURES = {
    'hp_is_present':         'Human present — VLM (share)',
    'opencv_has_face':        'Face detected — OpenCV (share)',
    'hp_canvas_coverage_pct': 'Human canvas coverage — VLM (%)',
    'opencv_face_ratio':      'Face-to-canvas ratio — OpenCV',
}

fig, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)
fig.suptitle('Face metrics over time: VLM vs OpenCV', fontsize=14, fontweight='bold')

for ax, (col, label) in zip(axes.flat, FACE_FEATURES.items()):
    for grp, color in PALETTE.items():
        sub = df[df['group_label'] == grp]
        ts = yearly_mean_ci(sub, col)
        if ts.empty:
            continue
        ax.plot(ts['year'], ts['mean'], marker='o', color=color, label=grp.capitalize(), linewidth=2, markersize=5)
        ax.fill_between(ts['year'], ts['lo'], ts['hi'], color=color, alpha=0.15)
    ax.set_title(label, fontsize=11)
    ax.set_xlabel('Year')
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.legend(framealpha=0.4)
    ax.grid(axis='y', linestyle='--', alpha=0.4)

fig.savefig(FIGURES / 'temporal_face_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

## 5 — Text presence & placement over time

- **Text presence rate**: share of thumbnails with any embedded text (derived from VLM `embedded_text` field).
- **Text function**: how the text is used (categorical share evolution).
- **Text placement zone**: which region of the canvas has the highest edge density (`edge_dominant_zone` from OpenCV).

In [ ]:
# ── Text presence rate ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))

for grp, color in PALETTE.items():
    sub = df[df['group_label'] == grp]
    ts = yearly_mean_ci(sub, 'has_text')
    if ts.empty:
        continue
    ax.plot(ts['year'], ts['mean'], marker='o', color=color, label=grp.capitalize(), linewidth=2, markersize=5)
    ax.fill_between(ts['year'], ts['lo'], ts['hi'], color=color, alpha=0.15)

ax.set_title('Share of thumbnails with embedded text over time', fontsize=12, fontweight='bold')
ax.set_xlabel('Year')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.legend(framealpha=0.4)
ax.grid(axis='y', linestyle='--', alpha=0.4)
fig.tight_layout()
fig.savefig(FIGURES / 'temporal_text_presence.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Text function share evolution ───────────────────────────────────────────
TEXT_FUNC_ORDER = ['factual_statement', 'question', 'hyperbolic_hook', 'label_or_acronym', 'none']

groups = ['independent', 'institutional']
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True, constrained_layout=True)
fig.suptitle('Text function share over time', fontsize=13, fontweight='bold')

cmap = plt.get_cmap('tab10')
tf_colors = {c: cmap(i) for i, c in enumerate(TEXT_FUNC_ORDER + ['other'])}

for ax, grp in zip(axes, groups):
    sub = df[df['group_label'] == grp].copy()
    sub['tf_clean'] = sub['text_function'].where(sub['text_function'].isin(TEXT_FUNC_ORDER), other='other')
    pivot = (
        sub.groupby(['year', 'tf_clean'])
           .size().unstack(fill_value=0)
           .pipe(lambda t: t.div(t.sum(axis=1), axis=0))
    )
    plot_cols = [c for c in TEXT_FUNC_ORDER + ['other'] if c in pivot.columns]
    pivot[plot_cols].reindex(range(YEAR_MIN, YEAR_MAX + 1)).fillna(0).plot.area(
        ax=ax, color=[tf_colors[c] for c in plot_cols], alpha=0.85, legend=False
    )
    ax.set_title(grp.capitalize(), fontsize=11)
    ax.set_xlabel('Year')
    ax.set_ylabel('Share of thumbnails')
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.grid(axis='y', linestyle='--', alpha=0.3)

handles = [plt.Rectangle((0,0),1,1, color=tf_colors[c]) for c in plot_cols]
fig.legend(handles, plot_cols, loc='lower center', ncol=len(plot_cols),
           bbox_to_anchor=(0.5, -0.06), framealpha=0.4, fontsize=9)
fig.savefig(FIGURES / 'temporal_text_function.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Text placement zone (edge-dominant region) ──────────────────────────────
ZONE_ORDER = ['top-left', 'top-center', 'top-right',
              'bottom-left', 'bottom-center', 'bottom-right']

fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True, constrained_layout=True)
fig.suptitle('Text/edge placement zone over time (OpenCV)', fontsize=13, fontweight='bold')

zone_cmap = plt.get_cmap('Set2')
zone_colors = {c: zone_cmap(i) for i, c in enumerate(ZONE_ORDER + ['other'])}

for ax, grp in zip(axes, groups):
    sub = df[df['group_label'] == grp].copy()
    sub['zone_clean'] = sub['edge_dominant_zone'].where(sub['edge_dominant_zone'].isin(ZONE_ORDER), other='other')
    pivot = (
        sub.groupby(['year', 'zone_clean'])
           .size().unstack(fill_value=0)
           .pipe(lambda t: t.div(t.sum(axis=1), axis=0))
    )
    plot_cols = [c for c in ZONE_ORDER + ['other'] if c in pivot.columns]
    pivot[plot_cols].reindex(range(YEAR_MIN, YEAR_MAX + 1)).fillna(0).plot.area(
        ax=ax, color=[zone_colors[c] for c in plot_cols], alpha=0.85, legend=False
    )
    ax.set_title(grp.capitalize(), fontsize=11)
    ax.set_xlabel('Year')
    ax.set_ylabel('Share of thumbnails')
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.grid(axis='y', linestyle='--', alpha=0.3)

handles = [plt.Rectangle((0,0),1,1, color=zone_colors[c]) for c in plot_cols]
fig.legend(handles, plot_cols, loc='lower center', ncol=len(plot_cols),
           bbox_to_anchor=(0.5, -0.06), framealpha=0.4, fontsize=9)
fig.savefig(FIGURES / 'temporal_text_zone.png', dpi=150, bbox_inches='tight')
plt.show()

## 6 — Categorical share evolution: archetype, visual style, emotion

In [ ]:
def top_categories(series, n=5, min_share=0.03):
    counts = series.dropna().value_counts(normalize=True)
    return counts[counts >= min_share].head(n).index.tolist()


def plot_cat_evolution(col, title, top_n=5, min_share=0.03, fname=None):
    cats = top_categories(df[col], n=top_n, min_share=min_share)
    if not cats:
        print(f"No categories found for {col}")
        return

    fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True, constrained_layout=True)
    fig.suptitle(title, fontsize=13, fontweight='bold')
    cmap = plt.get_cmap('tab10')
    cat_colors = {c: cmap(i) for i, c in enumerate(cats)}

    for ax, grp in zip(axes, ['independent', 'institutional']):
        sub = df[df['group_label'] == grp].copy()
        sub[col] = sub[col].where(sub[col].isin(cats), other='other')
        pivot = (
            sub.groupby(['year', col]).size().unstack(fill_value=0)
               .pipe(lambda t: t.div(t.sum(axis=1), axis=0))
        )
        plot_cols = [c for c in cats if c in pivot.columns]
        if 'other' in pivot.columns:
            plot_cols.append('other')
        pivot[plot_cols].reindex(range(YEAR_MIN, YEAR_MAX + 1)).fillna(0).plot.area(
            ax=ax, color=[cat_colors.get(c, '#cccccc') for c in plot_cols], alpha=0.85, legend=False
        )
        ax.set_title(grp.capitalize(), fontsize=11)
        ax.set_xlabel('Year')
        ax.set_ylabel('Share of thumbnails')
        ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
        ax.grid(axis='y', linestyle='--', alpha=0.3)

    handles = [plt.Rectangle((0,0),1,1, color=cat_colors.get(c, '#cccccc')) for c in plot_cols]
    fig.legend(handles, plot_cols, loc='lower center', ncol=len(plot_cols),
               bbox_to_anchor=(0.5, -0.06), framealpha=0.4, fontsize=9)
    if fname:
        fig.savefig(FIGURES / fname, dpi=150, bbox_inches='tight')
    plt.show()


plot_cat_evolution('archetype',                 'Archetype share over time',              top_n=5, fname='temporal_archetype.png')
plot_cat_evolution('visual_presentation_style', 'Visual presentation style over time',    top_n=4, fname='temporal_visual_style.png')
plot_cat_evolution('hp_dominant_emotion',       'Dominant emotion share over time',       top_n=5, fname='temporal_emotion.png')

## 7 — Group divergence heatmap

For each year: absolute mean difference between groups across all continuous features (VLM + OpenCV), z-scored for comparability.

In [ ]:
DIVERG_FEATURES = {
    # VLM
    'cb_sensationalism_score': 'Sensationalism',
    'hp_canvas_coverage_pct':  'Human canvas % (VLM)',
    'hp_is_present':           'Has human (VLM)',
    'cb_has_red_arrow':        'Red arrow/circle',
    'cb_has_exag_contrast':    'Exag. contrast',
    'has_text':                'Has embedded text',
    # OpenCV face
    'opencv_has_face':         'Face detected (OpenCV)',
    'opencv_face_ratio':       'Face/canvas ratio (OpenCV)',
    # OpenCV aesthetics
    'brightness':              'Brightness',
    'saturation':              'Saturation',
    'colorfulness':            'Colourfulness',
    'edge_density':            'Edge density',
}

years = list(range(YEAR_MIN, YEAR_MAX + 1))
rows = []
for year in years:
    sub = df[df['year'] == year]
    row = {'year': year}
    for feat in DIVERG_FEATURES:
        ind = sub[sub['group_label'] == 'independent'][feat].dropna()
        ins = sub[sub['group_label'] == 'institutional'][feat].dropna()
        row[feat] = abs(ind.mean() - ins.mean()) if (len(ind) > 1 and len(ins) > 1) else np.nan
    rows.append(row)

diverg = pd.DataFrame(rows).set_index('year')
diverg_z = diverg.apply(lambda c: (c - c.mean()) / c.std(), axis=0)
diverg_z.rename(columns=DIVERG_FEATURES, inplace=True)

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(
    diverg_z.T, ax=ax, cmap='YlOrRd',
    linewidths=0.4, linecolor='white',
    cbar_kws={'label': 'Normalised |Δ mean| (z-score)'},
)
ax.set_title('Group divergence per feature over time', fontsize=13, fontweight='bold')
ax.set_xlabel('Year')
plt.tight_layout()
fig.savefig(FIGURES / 'temporal_divergence_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 8 — Statistical test: early vs late period shift

Mann-Whitney U test per feature comparing early (2012–2016) vs late (2019–2025) period, separately for each group. Tests whether thumbnail aesthetics and semantics shifted significantly within each group over time.

In [ ]:
records = []
for grp in ['independent', 'institutional']:
    sub   = df[df['group_label'] == grp]
    early = sub[(sub['year'] >= EARLY[0]) & (sub['year'] <= EARLY[1])]
    late  = sub[(sub['year'] >= LATE[0])  & (sub['year'] <= LATE[1])]
    for feat, label in DIVERG_FEATURES.items():
        e_vals = early[feat].dropna()
        l_vals = late[feat].dropna()
        if len(e_vals) < 5 or len(l_vals) < 5:
            continue
        _, p = stats.mannwhitneyu(e_vals, l_vals, alternative='two-sided')
        records.append({
            'group':      grp,
            'feature':    label,
            'mean_early': round(e_vals.mean(), 3),
            'mean_late':  round(l_vals.mean(), 3),
            'delta':      round(l_vals.mean() - e_vals.mean(), 3),
            'p_value':    round(p, 4),
            'sig':        '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ('†' if p < 0.1 else ''))),
        })

shift_df = pd.DataFrame(records)
shift_df.to_csv(TABLES / 'temporal_shift_tests.csv', index=False)
print(shift_df.to_string(index=False))

## 9 — Archetype transition: early vs late stacked bars

In [ ]:
KNOWN_ARCHETYPES = [
    'Institutional Stock Photo',
    'Neutral Full-Frame Presenter',
    'Intense Close-Up Hook',
    'Minimalist Factual Slide',
    'Presenter + Statement Caption',
]

periods = {'Early\n(2012–2016)': EARLY, 'Late\n(2019–2025)': LATE}

fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
fig.suptitle('Archetype composition: early vs late period', fontsize=13, fontweight='bold')

cmap = plt.get_cmap('Set2')
arch_colors = {a: cmap(i) for i, a in enumerate(KNOWN_ARCHETYPES + ['other'])}

for ax, grp in zip(axes, ['independent', 'institutional']):
    sub = df[df['group_label'] == grp].copy()
    sub['arch_clean'] = sub['archetype'].where(sub['archetype'].isin(KNOWN_ARCHETYPES), other='other')

    period_shares = {}
    for label, (y0, y1) in periods.items():
        period_shares[label] = (
            sub[(sub['year'] >= y0) & (sub['year'] <= y1)]['arch_clean']
            .value_counts(normalize=True)
        )

    share_df = pd.DataFrame(period_shares).fillna(0)
    share_df = share_df.reindex([a for a in KNOWN_ARCHETYPES + ['other'] if a in share_df.index])

    bottom = np.zeros(len(share_df.columns))
    x = np.arange(len(share_df.columns))
    for arch in share_df.index:
        vals = share_df.loc[arch].values
        ax.bar(x, vals, bottom=bottom, color=arch_colors.get(arch, '#ccc'), label=arch, width=0.5)
        bottom += vals

    ax.set_xticks(x)
    ax.set_xticklabels(share_df.columns, fontsize=10)
    ax.set_title(grp.capitalize(), fontsize=11)
    ax.set_ylabel('Share of thumbnails')
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.grid(axis='y', linestyle='--', alpha=0.3)

handles = [plt.Rectangle((0,0),1,1, color=arch_colors[a]) for a in share_df.index]
fig.legend(handles, list(share_df.index), loc='lower center', ncol=3,
           bbox_to_anchor=(0.5, -0.12), framealpha=0.4, fontsize=9)

fig.savefig(FIGURES / 'temporal_archetype_early_late.png', dpi=150, bbox_inches='tight')
plt.show()